# New Tie Formation Across Community Boundaries (R2-1.1)

Tests a structural implication of the dyadic mechanism proposed in Section 2.3: whether accounts formed new ties across community boundaries more often after the narrative shift. A tie is new when the two accounts had not interacted earlier in either direction. Only ties between accounts already active in the baseline window are counted, so that newly arriving accounts do not inflate the count. Community membership is the canonical Louvain partition used throughout the manuscript.

1. post-shift: new ties (28 August to 29 September) among accounts active 1 to 26 August;
2. pre-shift benchmark: new ties in the second half of the pre-shift window (13 to 26 August) among accounts active 1 to 12 August;
3. robustness: equal seven-day windows on each side of the shift.

Also reproduces the comparison of new against repeated ties, which is not used as evidence because a partition built from the full period places repeated pairs in the same community by construction.

Input: `derived-data/hashed/temporal_edges_with_community_hashed.csv`. Reported in Section 5.7.

In [1]:
import numpy as np, pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from scipy.stats.contingency import odds_ratio
ev = pd.read_csv('temporal_edges_with_community_hashed.csv')   # derived-data/hashed/
ev['date_day'] = pd.to_datetime(ev['date_day'])
def win(a, b): return ev[(ev['date_day'] >= a) & (ev['date_day'] <= b)]
def new_ties(base, later):
    active = set(base['source']) | set(base['target'])
    old = set(zip(base['source'], base['target'])) | set(zip(base['target'], base['source']))
    t = later.groupby(['source', 'target'])['is_cross_community'].first().reset_index()
    t = t[t['source'].isin(active) & t['target'].isin(active)]
    t['new'] = [(s, d) not in old for s, d in zip(t['source'], t['target'])]
    return t

In [2]:
# (1)-(2) cross-community share of new ties, post-shift against the pre-shift benchmark
post = new_ties(win('2025-08-01', '2025-08-26'), win('2025-08-28', '2025-09-29')); post_new = post[post['new']]
pre  = new_ties(win('2025-08-01', '2025-08-12'), win('2025-08-13', '2025-08-26')); pre_new = pre[pre['new']]
a, n1 = int(post_new['is_cross_community'].sum()), len(post_new)
b, n0 = int(pre_new['is_cross_community'].sum()), len(pre_new)
tab = np.array([[a, n1 - a], [b, n0 - b]])
o = odds_ratio(tab); ci = o.confidence_interval(0.95); chi2, p, _, _ = chi2_contingency(tab, correction=False)
print(f'post-shift new ties crossing a boundary: {a}/{n1} = {100*a/n1:.1f}%')
print(f'pre-shift benchmark:                     {b}/{n0} = {100*b/n0:.1f}%')
print(f'OR = {o.statistic:.2f}, 95% CI [{ci.low:.2f}, {ci.high:.2f}], chi2 = {chi2:.2f}, p = {p:.3f}')

post-shift new ties crossing a boundary: 386/706 = 54.7%
pre-shift benchmark:                     132/224 = 58.9%
OR = 0.84, 95% CI [0.61, 1.15], chi2 = 1.25, p = 0.264


In [3]:
# (3) robustness: equal seven-day windows on each side of the shift
p7 = new_ties(win('2025-08-01', '2025-08-26'), win('2025-08-28', '2025-09-03')); p7 = p7[p7['new']]
b7 = new_ties(win('2025-08-01', '2025-08-19'), win('2025-08-20', '2025-08-26')); b7 = b7[b7['new']]
print(f"7 days before the shift: {int(b7.is_cross_community.sum())}/{len(b7)} = {100*b7.is_cross_community.mean():.1f}% | "
      f"7 days after: {int(p7.is_cross_community.sum())}/{len(p7)} = {100*p7.is_cross_community.mean():.1f}%")

7 days before the shift: 114/206 = 55.3% | 7 days after: 285/525 = 54.3%


In [4]:
# (4) new against repeated post-shift ties (not used as evidence, see note above)
t = pd.crosstab(post['is_cross_community'], post['new'])
tab = np.array([[t.loc[True, True], t.loc[True, False]], [t.loc[False, True], t.loc[False, False]]])
o = odds_ratio(tab); ci = o.confidence_interval(0.95)
print(f'new share among cross-community ties {100*tab[0,0]/tab[0].sum():.1f}% ({tab[0,0]}/{tab[0].sum()}), '
      f'among within-community ties {100*tab[1,0]/tab[1].sum():.1f}% ({tab[1,0]}/{tab[1].sum()}) | '
      f'OR = {o.statistic:.2f} [{ci.low:.2f}, {ci.high:.2f}], Fisher p = {fisher_exact(tab).pvalue:.1e}')
print(f'repeated pairs placed in the same community: {tab[1,1]} of {tab[0,1] + tab[1,1]}')

new share among cross-community ties 92.6% (386/417), among within-community ties 71.0% (320/451) | OR = 5.09 [3.31, 8.01], Fisher p = 4.8e-17
repeated pairs placed in the same community: 131 of 162
